# YOLO11n Training Parking Space Occupancy Detection

**Notebook:** `02_train_yolo11.ipynb`
**Project:** Parking Space Occupancy Detection (PKLot → YOLO11n → FastAPI deployment)
**Continues from:** `01_dataset_quality.ipynb`

## Prerequisites

This notebook assumes `01_dataset_quality.ipynb` has already run against `data/` and that:

- Images and labels have been validated (no missing pairs, no corrupt files).
- Labels have been remapped to the final 2-class schema:
  - `0` → `space-empty`
  - `1` → `space-occupied`
- `data/data.yaml` reflects the 2-class schema and points at the `train` / `valid` / `test`
  image directories.

If any of that hasn't happened yet, go back and run `01_dataset_quality.ipynb` first
this notebook does a lightweight sanity check (Section 6) but does not re-validate or
re-clean the dataset.

## Project Goal

Train a YOLO11n detector that meets:

- **mAP@50-95 ≥ 95%**
- **Inference latency ≤ 200 ms** (single-image, end-to-end)

## Training Workflow

1. Verify environment, reproducibility, and dataset integrity.
2. Initialize YOLO11n from pretrained COCO weights.
3. Fine-tune on PKLot with cosine LR scheduling, AMP, and early stopping.
4. Review training curves and validation metrics (P/R/F1/mAP50/mAP50-95).
5. Run error analysis (false positives/negatives, worst predictions, confidence spread).
6. Export the trained model to TorchScript and ONNX (TensorRT optional).
7. Benchmark inference latency on CPU and GPU, split into pre/inference/post stages.
8. Persist a Markdown training report and a machine-readable `experiment.json`.

## Experiment Tracking

Ultralytics writes native run artifacts (weights, curves, confusion matrix, CSV logs)
under `PROJECT_DIR / EXPERIMENT_NAME`. TensorBoard logging is enabled by default and can
be viewed with:

```bash
tensorboard --logdir runs/train
```

This notebook layers a structured `experiment.json` and `reports/training_report.md` on
top of the native Ultralytics outputs, so results stay reproducible and diffable even
outside of TensorBoard.

## Expected Outputs

- `runs/detect/runs/train/<experiment_name>/weights/best.pt` and `last.pt`
- `runs/detect/train/<experiment_name>/` curves, confusion matrix, PR curves (native Ultralytics)
- `<experiment_name>.torchscript` and `<experiment_name>.onnx` exported models
- `reports/training_report.md` human-readable summary
- `reports/experiment.json` machine-readable metadata + metrics


## 2. Imports

In [1]:
"""Core imports for the YOLO11n training pipeline."""
from __future__ import annotations

import json
import logging
import platform
import random
import subprocess
import time
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import yaml
from tqdm.auto import tqdm
from ultralytics import YOLO

%matplotlib inline
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 10


C:\Users\user\Projects\Shelf Product Detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Configuration

In [2]:
@dataclass
class TrainingConfig:
    """Central configuration for YOLO11n training on PKLot.

    Parameters
    ----------
    model_weights : str
        Pretrained checkpoint to fine-tune from.
    data_yaml : Path
        Path to the dataset's ``data.yaml`` (produced by notebook 01).
    image_size : int
        Training/inference image size (square, pixels).
    epochs : int
        Maximum training epochs.
    batch_size : int
        Batch size. Use ``-1`` to let Ultralytics auto-select based on GPU memory.
    workers : int
        Dataloader worker processes.
    patience : int
        Early-stopping patience, in epochs without mAP improvement.
    optimizer : str
        Optimizer name understood by Ultralytics (``"SGD"``, ``"Adam"``, ``"AdamW"``, ...).
    learning_rate : float
        Initial learning rate (``lr0``).
    final_lr_fraction : float
        Final LR as a fraction of ``learning_rate`` (``lrf``), used by the cosine schedule.
    weight_decay : float
        Optimizer weight decay.
    cosine_lr : bool
        Whether to use cosine LR annealing instead of linear decay.
    augmentation : Dict[str, float]
        Ultralytics augmentation hyperparameters (mosaic, mixup, hsv, flip, etc.).
    project_dir : Path
        Root directory Ultralytics writes run artifacts under.
    experiment_name : str
        Sub-directory name for this run (``project_dir / experiment_name``).
    seed : int
        Global random seed.
    device : str
        Training device string (``"0"`` for first GPU, ``"cpu"``, ``"0,1"`` for multi-GPU).
    amp : bool
        Whether to use automatic mixed precision.
    resume : bool
        Whether to resume from the last checkpoint in ``experiment_name``, if present.
    """

    model_weights: str = "yolo11n.pt"
    data_yaml: Path = Path("data/data.yaml")
    image_size: int = 640
    epochs: int = 150
    batch_size: int = 16
    workers: int = 8
    patience: int = 30
    optimizer: str = "AdamW"
    learning_rate: float = 1e-3
    final_lr_fraction: float = 0.01
    weight_decay: float = 5e-4
    cosine_lr: bool = True
    augmentation: Dict[str, float] = field(
        default_factory=lambda: {
            "mosaic": 1.0,
            "mixup": 0.1,
            "hsv_h": 0.015,
            "hsv_s": 0.7,
            "hsv_v": 0.4,
            "degrees": 0.0,
            "translate": 0.1,
            "scale": 0.5,
            "shear": 0.0,
            "fliplr": 0.5,
            "flipud": 0.0,
        }
    )
    project_dir: Path = Path("runs/train")
    experiment_name: str = "pklot_yolo11n"
    seed: int = 42
    device: str = "0"
    amp: bool = True
    resume: bool = True


CFG = TrainingConfig()
REPORT_DIR = Path("reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)
CFG


TrainingConfig(model_weights='yolo11n.pt', data_yaml=WindowsPath('data/data.yaml'), image_size=640, epochs=150, batch_size=16, workers=8, patience=30, optimizer='AdamW', learning_rate=0.001, final_lr_fraction=0.01, weight_decay=0.0005, cosine_lr=True, augmentation={'mosaic': 1.0, 'mixup': 0.1, 'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4, 'degrees': 0.0, 'translate': 0.1, 'scale': 0.5, 'shear': 0.0, 'fliplr': 0.5, 'flipud': 0.0}, project_dir=WindowsPath('runs/train'), experiment_name='pklot_yolo11n', seed=42, device='0', amp=True, resume=True)

## 4. Environment Check

In [3]:
def setup_logging(log_level: int = logging.INFO) -> logging.Logger:
    """Configure and return the project logger.

    Parameters
    ----------
    log_level : int
        Logging verbosity level.

    Returns
    -------
    logging.Logger
        Configured logger instance, safe to re-call across cell re-executions.
    """
    logger = logging.getLogger("pklot_train_yolo11")
    logger.setLevel(log_level)
    logger.handlers.clear()

    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(message)s",
        datefmt="%H:%M:%S",
    )
    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)
    logger.propagate = False
    return logger


logger = setup_logging()


def display_environment_info() -> Dict[str, Any]:
    """Collect and log environment / hardware information.

    Returns
    -------
    Dict[str, Any]
        Python, PyTorch, Ultralytics, and CUDA/GPU details.
    """
    import ultralytics

    info: Dict[str, Any] = {
        "python_version": platform.python_version(),
        "torch_version": torch.__version__,
        "ultralytics_version": ultralytics.__version__,
        "cuda_available": torch.cuda.is_available(),
    }

    if info["cuda_available"]:
        info["cuda_version"] = torch.version.cuda
        info["gpu_name"] = torch.cuda.get_device_name(0)
        info["gpu_memory_gb"] = round(
            torch.cuda.get_device_properties(0).total_memory / (1024**3), 2
        )
        info["gpu_count"] = torch.cuda.device_count()
    else:
        info["cuda_version"] = None
        info["gpu_name"] = None
        info["gpu_memory_gb"] = None
        info["gpu_count"] = 0

    for key, value in info.items():
        logger.info(f"{key}: {value}")
    return info


ENV_INFO = display_environment_info()

if not ENV_INFO["cuda_available"] and CFG.device != "cpu":
    logger.warning("CUDA not available; falling back to CPU. Training will be slower.")
    CFG.device = "cpu"
    CFG.amp = False


17:27:54 | INFO     | python_version: 3.11.9
17:27:54 | INFO     | torch_version: 2.13.0+cu130
17:27:54 | INFO     | ultralytics_version: 8.4.115
17:27:54 | INFO     | cuda_available: True
17:27:54 | INFO     | cuda_version: 13.0
17:27:54 | INFO     | gpu_name: NVIDIA GeForce RTX 5060
17:27:54 | INFO     | gpu_memory_gb: 7.96
17:27:54 | INFO     | gpu_count: 1


## 5. Reproducibility

In [4]:
def seed_everything(seed: int) -> None:
    """Seed all relevant random number generators for reproducible training.

    Parameters
    ----------
    seed : int
        Seed value applied to ``random``, ``numpy``, and ``torch`` (CPU + CUDA).
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    logger.info(f"Global seed set to {seed}.")


seed_everything(CFG.seed)


17:27:54 | INFO     | Global seed set to 42.


## 6. Dataset Verification

In [5]:
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import yaml

@dataclass
class DatasetCheckResult:
    data_yaml_exists: bool
    n_classes: Optional[int]
    class_names: Optional[List[str]]
    classes_match_expected: bool
    split_image_counts: Dict[str, int]
    split_label_counts: Dict[str, int]
    ok: bool


def verify_dataset(cfg: TrainingConfig, expected_class_names: Tuple[str, ...]) -> DatasetCheckResult:
    """Verify data.yaml and dataset folder structure before training."""

    if not cfg.data_yaml.is_file():
        logger.error(f"data.yaml not found: {cfg.data_yaml}")
        return DatasetCheckResult(False, None, None, False, {}, {}, False)

    with cfg.data_yaml.open("r") as f:
        data_cfg = yaml.safe_load(f)

    n_classes = data_cfg.get("nc")
    class_names = data_cfg.get("names", [])
    classes_match_expected = tuple(class_names) == expected_class_names

    dataset_root = cfg.data_yaml.parent  # data/

    split_image_counts = {}
    split_label_counts = {}

    for split_name, yaml_key in (
        ("train", "train"),
        ("valid", "val"),
        ("test", "test"),
    ):
        rel_path = data_cfg.get(yaml_key)
        if rel_path is None:
            split_image_counts[split_name] = 0
            split_label_counts[split_name] = 0
            continue

        images_dir = dataset_root / rel_path
        labels_dir = images_dir.parent / "labels"

        image_count = (
            sum(1 for p in images_dir.iterdir()
                if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
            if images_dir.is_dir() else 0
        )

        label_count = (
            sum(1 for p in labels_dir.iterdir()
                if p.suffix.lower() == ".txt")
            if labels_dir.is_dir() else 0
        )

        split_image_counts[split_name] = image_count
        split_label_counts[split_name] = label_count

        logger.info(f"{split_name}: {image_count} images, {label_count} labels")

    ok = (
        n_classes == len(expected_class_names)
        and classes_match_expected
        and split_image_counts["train"] > 0
        and split_image_counts["valid"] > 0
    )

    if ok:
        logger.info("Dataset verification passed.")
    else:
        logger.error("Dataset verification failed.")

    return DatasetCheckResult(
        data_yaml_exists=True,
        n_classes=n_classes,
        class_names=class_names,
        classes_match_expected=classes_match_expected,
        split_image_counts=split_image_counts,
        split_label_counts=split_label_counts,
        ok=ok,
    )


EXPECTED_CLASS_NAMES = ("space-empty", "space-occupied")
DATASET_CHECK = verify_dataset(CFG, EXPECTED_CLASS_NAMES)

assert DATASET_CHECK.ok, (
    "Dataset verification failed. Fix the dataset before training."
)

17:27:54 | INFO     | train: 8691 images, 8691 labels
17:27:54 | INFO     | valid: 2483 images, 2483 labels
17:27:54 | INFO     | test: 1242 images, 1242 labels
17:27:54 | INFO     | Dataset verification passed.


## 7. Baseline Training Configuration

Hyperparameter choices for this baseline run, and why:

| Hyperparameter | Value | Rationale |
|---|---|---|
| `model_weights` | `yolo11n.pt` | Smallest YOLO11 variant fastest inference, best fit for a ≤200ms latency budget on likely CPU/edge deployment. |
| `image_size` | 640 | Ultralytics default; PKLot parking-space boxes are small and dense, so this preserves enough resolution without blowing up latency. |
| `epochs` | 150 | Generous ceiling; early stopping (`patience=30`) is expected to trigger well before this on a dataset PKLot's size. |
| `batch_size` | 16 | Fits comfortably on a single mid-range GPU (≥8GB) at 640px for YOLO11n; lower if OOM, raise if GPU memory allows. |
| `optimizer` | `AdamW` | Converges faster than SGD on smaller/less noisy detection datasets like PKLot; weight decay decoupled from LR. |
| `learning_rate` (`lr0`) | 1e-3 | Standard AdamW starting point for YOLO fine-tuning from COCO weights. |
| `final_lr_fraction` (`lrf`) | 0.01 | Cosine schedule anneals down to 1% of the initial LR by the final epoch. |
| `weight_decay` | 5e-4 | Mild regularization; PKLot is visually repetitive (fixed camera angles), so overfitting risk is real. |
| `cosine_lr` | `True` | Smoother convergence and typically better final mAP than linear/step decay for short-to-medium fine-tuning runs. |
| `patience` | 30 | Stops training if mAP50-95 hasn't improved in 30 epochs, avoiding wasted compute once the model plateaus. |
| `amp` | `True` (GPU only) | Mixed precision roughly halves memory and training time with negligible accuracy impact on modern GPUs. |
| Augmentation | mosaic=1.0, mixup=0.1, moderate HSV/translate/scale, `fliplr=0.5`, `flipud=0.0` | Parking-lot cameras are always upright, so vertical flips would create unrealistic orientations disabled. Horizontal flip, HSV jitter, and moderate scale/translate improve robustness to lighting and camera-angle variation across lots without distorting box geometry. |

These are starting-point defaults, not final tuned values Section 9 trains against
them, and Sections 10–11 are where you'd notice if any of these need revisiting
(e.g. LR too high if loss is unstable, too much augmentation if train/val gap is large).

In [6]:
logger.info("Baseline training configuration:")
for key, value in asdict(CFG).items():
    logger.info(f"  {key}: {value}")


17:27:54 | INFO     | Baseline training configuration:
17:27:54 | INFO     |   model_weights: yolo11n.pt
17:27:54 | INFO     |   data_yaml: data\data.yaml
17:27:54 | INFO     |   image_size: 640
17:27:54 | INFO     |   epochs: 150
17:27:54 | INFO     |   batch_size: 16
17:27:54 | INFO     |   workers: 8
17:27:54 | INFO     |   patience: 30
17:27:54 | INFO     |   optimizer: AdamW
17:27:54 | INFO     |   learning_rate: 0.001
17:27:54 | INFO     |   final_lr_fraction: 0.01
17:27:54 | INFO     |   weight_decay: 0.0005
17:27:54 | INFO     |   cosine_lr: True
17:27:54 | INFO     |   augmentation: {'mosaic': 1.0, 'mixup': 0.1, 'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4, 'degrees': 0.0, 'translate': 0.1, 'scale': 0.5, 'shear': 0.0, 'fliplr': 0.5, 'flipud': 0.0}
17:27:54 | INFO     |   project_dir: runs\train
17:27:54 | INFO     |   experiment_name: pklot_yolo11n
17:27:54 | INFO     |   seed: 42
17:27:54 | INFO     |   device: 0
17:27:54 | INFO     |   amp: True
17:27:54 | INFO     |   resume:

## 8. Model Initialization

In [7]:
def initialize_model(model_weights: str) -> YOLO:
    """Load a YOLO model and log its architecture summary.

    Parameters
    ----------
    model_weights : str
        Path or identifier of the pretrained weights to load
        (e.g. ``"yolo11n.pt"``); downloaded automatically by Ultralytics
        if not already cached locally.

    Returns
    -------
    YOLO
        Loaded Ultralytics ``YOLO`` model instance.
    """
    model = YOLO(model_weights)
    n_params = sum(p.numel() for p in model.model.parameters())
    n_trainable = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
    logger.info(f"Loaded {model_weights}")
    logger.info(f"Total parameters: {n_params:,}")
    logger.info(f"Trainable parameters: {n_trainable:,}")
    return model


MODEL = initialize_model(CFG.model_weights)
MODEL.info(detailed=False)


17:27:54 | INFO     | Loaded yolo11n.pt
17:27:54 | INFO     | Total parameters: 2,624,080
17:27:54 | INFO     | Trainable parameters: 0


YOLO11n summary: 181 layers, 2,624,080 parameters, 0 gradients, 6.7 GFLOPs


(181, 2624080, 0, 6.6733184)

## 9. Training

Launches Ultralytics training with AMP, cosine LR, early stopping, and periodic
checkpointing. TensorBoard logging is on by default (`runs/train/<experiment_name>`).
Set `CFG.resume = True` before running this cell to resume from the last checkpoint of
the same `experiment_name` instead of starting fresh.

In [ ]:
def train_model(model: YOLO, cfg: TrainingConfig) -> Tuple[Any, float]:
    """Train a YOLO model using the given configuration.

    Parameters
    ----------
    model : YOLO
        Initialized (pretrained) Ultralytics model.
    cfg : TrainingConfig
        Training configuration.

    Returns
    -------
    Tuple[Any, float]
        Ultralytics training results object, and wall-clock training duration in seconds.
    """
    start = time.time()
    results = model.train(
        data=str(cfg.data_yaml),
        imgsz=cfg.image_size,
        epochs=cfg.epochs,
        batch=cfg.batch_size,
        workers=cfg.workers,
        patience=cfg.patience,
        optimizer=cfg.optimizer,
        lr0=cfg.learning_rate,
        lrf=cfg.final_lr_fraction,
        weight_decay=cfg.weight_decay,
        cos_lr=cfg.cosine_lr,
        mosaic=cfg.augmentation["mosaic"],
        mixup=cfg.augmentation["mixup"],
        hsv_h=cfg.augmentation["hsv_h"],
        hsv_s=cfg.augmentation["hsv_s"],
        hsv_v=cfg.augmentation["hsv_v"],
        degrees=cfg.augmentation["degrees"],
        translate=cfg.augmentation["translate"],
        scale=cfg.augmentation["scale"],
        shear=cfg.augmentation["shear"],
        fliplr=cfg.augmentation["fliplr"],
        flipud=cfg.augmentation["flipud"],
        project=str(cfg.project_dir),
        name=cfg.experiment_name,
        seed=cfg.seed,
        device=cfg.device,
        amp=cfg.amp,
        resume=cfg.resume,
        exist_ok=True,
        plots=True,
        save=True,
        save_period=10,
        val=True,
        verbose=True,
    )
    duration = time.time() - start
    logger.info(f"Training complete in {duration / 60:.1f} minutes.")
    return results, duration


TRAIN_RESULTS, TRAINING_DURATION_SEC = train_model(MODEL, CFG)

RUN_DIR = CFG.project_dir / CFG.experiment_name
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
LAST_WEIGHTS = RUN_DIR / "weights" / "last.pt"
logger.info(f"Run directory: {RUN_DIR}")
logger.info(f"Best weights:  {BEST_WEIGHTS} (exists={BEST_WEIGHTS.is_file()})")
logger.info(f"Last weights:  {LAST_WEIGHTS} (exists={LAST_WEIGHTS.is_file()})")


WARNING model 'yolo11n.pt' is not a resumable training checkpoint (missing epoch/optimizer state). Use 'resume' only to continue incomplete training. Starting new training instead.
Ultralytics 8.4.115  Python-3.11.9 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=data\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_widt

In [ ]:

RUN_DIR = CFG.project_dir / CFG.experiment_name
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
LAST_WEIGHTS = RUN_DIR / "weights" / "last.pt"
logger.info(f"Run directory: {RUN_DIR}")
logger.info(f"Best weights:  {BEST_WEIGHTS} (exists={BEST_WEIGHTS.is_file()})")
logger.info(f"Last weights:  {LAST_WEIGHTS} (exists={LAST_WEIGHTS.is_file()})")

## 10. Training Curves

In [ ]:
def load_training_history(run_dir: Path) -> pd.DataFrame:
    """Load the per-epoch training history CSV written by Ultralytics.

    Parameters
    ----------
    run_dir : Path
        Run directory (``project_dir / experiment_name``).

    Returns
    -------
    pd.DataFrame
        Parsed ``results.csv`` with whitespace-stripped column names.
    """
    csv_path = run_dir / "results.csv"
    if not csv_path.is_file():
        logger.warning(f"No results.csv found at {csv_path}")
        return pd.DataFrame()
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    return df


def plot_training_curves(history: pd.DataFrame) -> None:
    """Plot loss and validation metric curves across training epochs.

    Parameters
    ----------
    history : pd.DataFrame
        Output of ``load_training_history``.
    """
    if history.empty:
        logger.warning("No training history available; skipping curve plots.")
        return

    loss_cols = [c for c in history.columns if c.endswith("loss")]
    metric_cols = {
        "precision": next((c for c in history.columns if "precision" in c.lower()), None),
        "recall": next((c for c in history.columns if "recall" in c.lower()), None),
        "mAP50": next((c for c in history.columns if "map50" in c.lower() and "95" not in c.lower()), None),
        "mAP50-95": next((c for c in history.columns if "map50-95" in c.lower()), None),
    }

    fig, axes = plt.subplots(1, 2, figsize=(15, 5), dpi=110)

    for col in loss_cols:
        axes[0].plot(history["epoch"], history[col], label=col.replace("/", " ").strip())
    axes[0].set_title("Training / Validation Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.3)

    for label, col in metric_cols.items():
        if col and col in history.columns:
            axes[1].plot(history["epoch"], history[col], label=label)
    axes[1].set_title("Validation Metrics")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Score")
    axes[1].set_ylim(0, 1.05)
    axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


TRAINING_HISTORY = load_training_history(RUN_DIR)
plot_training_curves(TRAINING_HISTORY)


## 11. Validation

In [ ]:
def run_validation(model_weights: Path, data_yaml: Path, image_size: int, device: str) -> Dict[str, float]:
    """Run full validation and extract summary metrics.

    Parameters
    ----------
    model_weights : Path
        Path to the checkpoint to validate (typically ``best.pt``).
    data_yaml : Path
        Dataset yaml to validate against.
    image_size : int
        Validation image size.
    device : str
        Device string for validation.

    Returns
    -------
    Dict[str, float]
        Precision, recall, F1, mAP50, and mAP50-95, plus the native
        Ultralytics validation results object under ``"_raw"``.
    """
    val_model = YOLO(str(model_weights))
    val_results = val_model.val(data=str(data_yaml), imgsz=image_size, device=device, plots=True)

    precision = float(val_results.box.mp)
    recall = float(val_results.box.mr)
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    metrics = {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mAP50": float(val_results.box.map50),
        "mAP50-95": float(val_results.box.map),
        "_raw": val_results,
    }

    logger.info(
        f"Validation P={precision:.4f} R={recall:.4f} F1={f1:.4f} "
        f"mAP50={metrics['mAP50']:.4f} mAP50-95={metrics['mAP50-95']:.4f}"
    )
    target_met = metrics["mAP50-95"] >= 0.95
    logger.info(f"mAP50-95 >= 0.95 target: {'MET' if target_met else 'NOT MET'}")
    return metrics


VAL_METRICS = run_validation(BEST_WEIGHTS, CFG.data_yaml, CFG.image_size, CFG.device)

pd.DataFrame(
    {"metric": ["precision", "recall", "f1", "mAP50", "mAP50-95"],
     "value": [VAL_METRICS[k] for k in ("precision", "recall", "f1", "mAP50", "mAP50-95")]}
)


In [ ]:
# Ultralytics saves the confusion matrix and PR curve plots automatically during val()
# under RUN_DIR (or a new "val" run directory) display them inline here.
val_plot_dir = Path(VAL_METRICS["_raw"].save_dir)
for plot_name in ("confusion_matrix.png", "confusion_matrix_normalized.png", "PR_curve.png"):
    plot_path = val_plot_dir / plot_name
    if plot_path.is_file():
        img = plt.imread(plot_path)
        fig, ax = plt.subplots(figsize=(7, 6), dpi=110)
        ax.imshow(img)
        ax.set_title(plot_name)
        ax.axis("off")
        plt.show()
    else:
        logger.warning(f"Expected plot not found: {plot_path}")


## 12. Error Analysis

In [ ]:
@dataclass
class PredictionRecord:
    """A single per-image prediction summary used for error analysis."""

    filename: str
    n_true_boxes: int
    n_pred_boxes: int
    n_matched: int
    n_false_positives: int
    n_false_negatives: int
    mean_confidence: float
    min_confidence: float


def iou_xyxy(box_a: np.ndarray, box_b: np.ndarray) -> float:
    """Compute IoU between two ``[x1, y1, x2, y2]`` boxes.

    Parameters
    ----------
    box_a, box_b : np.ndarray
        Boxes in absolute pixel ``[x1, y1, x2, y2]`` format.

    Returns
    -------
    float
        Intersection-over-union in ``[0, 1]``.
    """
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_a = max(0.0, box_a[2] - box_a[0]) * max(0.0, box_a[3] - box_a[1])
    area_b = max(0.0, box_b[2] - box_b[0]) * max(0.0, box_b[3] - box_b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def analyze_predictions(
    model: YOLO, images_dir: Path, labels_dir: Path, image_size: int,
    iou_threshold: float = 0.5, conf_threshold: float = 0.25,
) -> pd.DataFrame:
    """Run inference on a validation split and summarize per-image errors.

    Parameters
    ----------
    model : YOLO
        Trained model (loaded from ``best.pt``).
    images_dir : Path
        Directory of images to analyze (typically the validation split).
    labels_dir : Path
        Corresponding YOLO label directory.
    image_size : int
        Inference image size.
    iou_threshold : float
        IoU threshold above which a prediction is considered matched to a ground-truth box.
    conf_threshold : float
        Confidence threshold applied during inference.

    Returns
    -------
    pd.DataFrame
        One row per image with true positive/false positive/false negative counts
        and confidence statistics. Images that fail to load are skipped and logged.
    """
    records: List[PredictionRecord] = []
    image_paths = sorted(
        p for p in images_dir.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp")
    )

    for img_path in tqdm(image_paths, desc="Error analysis", leave=False):
        try:
            result = model.predict(str(img_path), imgsz=image_size, conf=conf_threshold, verbose=False)[0]
        except Exception as exc:  # noqa: BLE001
            logger.warning(f"Inference failed on {img_path.name}: {exc}")
            continue

        pred_boxes = result.boxes.xyxy.cpu().numpy() if result.boxes is not None else np.empty((0, 4))
        pred_confs = result.boxes.conf.cpu().numpy() if result.boxes is not None else np.empty((0,))

        label_path = labels_dir / f"{img_path.stem}.txt"
        h_img, w_img = result.orig_shape
        true_boxes = []
        if label_path.is_file():
            for line in label_path.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                _, x, y, w, h = (float(v) if i > 0 else int(v) for i, v in enumerate(parts))
                x1 = (x - w / 2) * w_img
                y1 = (y - h / 2) * h_img
                x2 = (x + w / 2) * w_img
                y2 = (y + h / 2) * h_img
                true_boxes.append([x1, y1, x2, y2])
        true_boxes = np.array(true_boxes) if true_boxes else np.empty((0, 4))

        matched_true = set()
        matched_pred = set()
        for pi, pbox in enumerate(pred_boxes):
            best_iou, best_ti = 0.0, -1
            for ti, tbox in enumerate(true_boxes):
                if ti in matched_true:
                    continue
                iou = iou_xyxy(pbox, tbox)
                if iou > best_iou:
                    best_iou, best_ti = iou, ti
            if best_iou >= iou_threshold and best_ti >= 0:
                matched_true.add(best_ti)
                matched_pred.add(pi)

        n_matched = len(matched_pred)
        n_fp = len(pred_boxes) - n_matched
        n_fn = len(true_boxes) - len(matched_true)

        records.append(
            PredictionRecord(
                filename=img_path.name,
                n_true_boxes=len(true_boxes),
                n_pred_boxes=len(pred_boxes),
                n_matched=n_matched,
                n_false_positives=n_fp,
                n_false_negatives=n_fn,
                mean_confidence=float(pred_confs.mean()) if len(pred_confs) else float("nan"),
                min_confidence=float(pred_confs.min()) if len(pred_confs) else float("nan"),
            )
        )

    return pd.DataFrame([vars(r) for r in records])


with open(CFG.data_yaml) as _f:
    _data_cfg = yaml.safe_load(_f)
VAL_IMAGES_DIR = Path(_data_cfg["val"])
VAL_LABELS_DIR = Path(str(VAL_IMAGES_DIR).replace("images", "labels"))

BEST_MODEL = YOLO(str(BEST_WEIGHTS))
ERROR_ANALYSIS_DF = analyze_predictions(BEST_MODEL, VAL_IMAGES_DIR, VAL_LABELS_DIR, CFG.image_size)

print(f"Total false positives: {ERROR_ANALYSIS_DF['n_false_positives'].sum()}")
print(f"Total false negatives: {ERROR_ANALYSIS_DF['n_false_negatives'].sum()}")
print(f"Mean confidence: {ERROR_ANALYSIS_DF['mean_confidence'].mean():.4f}")

print("\nWorst predictions (most combined FP + FN):")
worst = ERROR_ANALYSIS_DF.assign(
    total_errors=ERROR_ANALYSIS_DF["n_false_positives"] + ERROR_ANALYSIS_DF["n_false_negatives"]
).sort_values("total_errors", ascending=False)
worst.head(10)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=110)

axes[0].hist(ERROR_ANALYSIS_DF["mean_confidence"].dropna(), bins=30, color="#4363D8")
axes[0].set_title("Prediction Confidence Distribution")
axes[0].set_xlabel("Mean confidence per image")
axes[0].set_ylabel("Images")

error_totals = ERROR_ANALYSIS_DF[["n_false_positives", "n_false_negatives"]].sum()
axes[1].bar(error_totals.index, error_totals.values, color=["#DC3C3C", "#F58231"])
axes[1].set_title("False Positives vs False Negatives")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()


## 13. Export

Exports the trained model to TorchScript and ONNX for deployment. TensorRT export is
attempted only if `tensorrt` is importable in this environment it's skipped
gracefully (with a log message) otherwise, since it requires an NVIDIA GPU + matching
CUDA/TensorRT install that may not be present everywhere this notebook runs.

In [ ]:
@dataclass
class ExportResult:
    """Outcome of exporting a model to one target format."""

    format: str
    success: bool
    path: Optional[str] = None
    error: Optional[str] = None


def export_model(model: YOLO, formats: List[str], image_size: int) -> List[ExportResult]:
    """Export a trained model to one or more deployment formats.

    Parameters
    ----------
    model : YOLO
        Trained model (loaded from ``best.pt``).
    formats : List[str]
        Ultralytics export format identifiers, e.g. ``["torchscript", "onnx", "engine"]``.
    image_size : int
        Export image size (should match training/inference size).

    Returns
    -------
    List[ExportResult]
        One result per requested format; failures are captured, not raised,
        so one unsupported format doesn't block the others.
    """
    results = []
    for fmt in formats:
        try:
            exported_path = model.export(format=fmt, imgsz=image_size)
            results.append(ExportResult(format=fmt, success=True, path=str(exported_path)))
            logger.info(f"Exported {fmt} -> {exported_path}")
        except Exception as exc:  # noqa: BLE001
            results.append(ExportResult(format=fmt, success=False, error=str(exc)))
            logger.warning(f"Export to {fmt} failed: {exc}")
    return results


EXPORT_FORMATS = ["torchscript", "onnx"]

try:
    import tensorrt  # noqa: F401

    EXPORT_FORMATS.append("engine")
    logger.info("tensorrt available; TensorRT export enabled.")
except ImportError:
    logger.info("tensorrt not available; skipping TensorRT export.")

EXPORT_RESULTS = export_model(BEST_MODEL, EXPORT_FORMATS, CFG.image_size)
pd.DataFrame([vars(r) for r in EXPORT_RESULTS])


## 14. Benchmark

In [ ]:
@dataclass
class LatencyBenchmark:
    """Latency benchmark result for one device, split into pipeline stages."""

    device: str
    n_runs: int
    preprocess_ms_mean: float
    inference_ms_mean: float
    postprocess_ms_mean: float
    total_ms_mean: float
    total_ms_p95: float
    meets_200ms_target: bool


def benchmark_latency(
    model_weights: Path, image_path: Path, image_size: int, device: str, n_warmup: int = 5, n_runs: int = 50
) -> LatencyBenchmark:
    """Benchmark per-stage inference latency on a given device.

    Parameters
    ----------
    model_weights : Path
        Path to the checkpoint to benchmark.
    image_path : Path
        A representative image used for repeated inference.
    image_size : int
        Inference image size.
    device : str
        Device string (``"cpu"`` or a CUDA device index like ``"0"``).
    n_warmup : int
        Number of warmup iterations excluded from timing.
    n_runs : int
        Number of timed iterations.

    Returns
    -------
    LatencyBenchmark
        Mean/percentile latency broken down by preprocessing, inference, and
        postprocessing stages, plus a pass/fail flag against the 200ms target.
    """
    bench_model = YOLO(str(model_weights))
    bench_model.to(device)

    for _ in range(n_warmup):
        bench_model.predict(str(image_path), imgsz=image_size, device=device, verbose=False)

    pre_times, inf_times, post_times, total_times = [], [], [], []
    for _ in tqdm(range(n_runs), desc=f"Benchmark[{device}]", leave=False):
        result = bench_model.predict(str(image_path), imgsz=image_size, device=device, verbose=False)[0]
        speed = result.speed  # dict with 'preprocess', 'inference', 'postprocess' in ms
        pre_times.append(speed["preprocess"])
        inf_times.append(speed["inference"])
        post_times.append(speed["postprocess"])
        total_times.append(speed["preprocess"] + speed["inference"] + speed["postprocess"])

    total_arr = np.array(total_times)
    result = LatencyBenchmark(
        device=device,
        n_runs=n_runs,
        preprocess_ms_mean=float(np.mean(pre_times)),
        inference_ms_mean=float(np.mean(inf_times)),
        postprocess_ms_mean=float(np.mean(post_times)),
        total_ms_mean=float(total_arr.mean()),
        total_ms_p95=float(np.percentile(total_arr, 95)),
        meets_200ms_target=bool(total_arr.mean() <= 200.0),
    )
    logger.info(
        f"[{device}] mean={result.total_ms_mean:.1f}ms p95={result.total_ms_p95:.1f}ms "
        f"(target<=200ms: {'MET' if result.meets_200ms_target else 'NOT MET'})"
    )
    return result


_sample_image = next(VAL_IMAGES_DIR.iterdir())

BENCHMARK_RESULTS: Dict[str, LatencyBenchmark] = {}
BENCHMARK_RESULTS["cpu"] = benchmark_latency(BEST_WEIGHTS, _sample_image, CFG.image_size, device="cpu")
if torch.cuda.is_available():
    BENCHMARK_RESULTS["gpu"] = benchmark_latency(BEST_WEIGHTS, _sample_image, CFG.image_size, device="0")
else:
    logger.info("No GPU available; skipping GPU benchmark.")

pd.DataFrame([vars(v) for v in BENCHMARK_RESULTS.values()])


## 15. Save Training Report

In [ ]:
def build_training_report_markdown(
    cfg: TrainingConfig,
    env_info: Dict[str, Any],
    val_metrics: Dict[str, float],
    training_duration_sec: float,
    weights_path: Path,
    benchmarks: Dict[str, LatencyBenchmark],
) -> str:
    """Render a Markdown training report.

    Parameters
    ----------
    cfg : TrainingConfig
        Training configuration used for this run.
    env_info : Dict[str, Any]
        Output of ``display_environment_info``.
    val_metrics : Dict[str, float]
        Output of ``run_validation`` (the ``"_raw"`` key is ignored).
    training_duration_sec : float
        Wall-clock training duration, in seconds.
    weights_path : Path
        Path to the best checkpoint.
    benchmarks : Dict[str, LatencyBenchmark]
        Per-device latency benchmark results.

    Returns
    -------
    str
        Complete Markdown report content.
    """
    model_size_mb = weights_path.stat().st_size / (1024 * 1024) if weights_path.is_file() else float("nan")
    hours, remainder = divmod(training_duration_sec, 3600)
    minutes, seconds = divmod(remainder, 60)

    lines = [
        "# Training Report PKLot YOLO11n Parking Space Occupancy Detector",
        "",
        f"Generated: {datetime.now(timezone.utc).isoformat()}",
        "",
        "## Hardware",
        "",
        f"- Python: {env_info['python_version']}",
        f"- PyTorch: {env_info['torch_version']}",
        f"- Ultralytics: {env_info['ultralytics_version']}",
        f"- CUDA available: {env_info['cuda_available']}",
        f"- GPU: {env_info.get('gpu_name') or 'N/A'}",
        f"- GPU memory: {env_info.get('gpu_memory_gb') or 'N/A'} GB",
        "",
        "## Configuration",
        "",
        "| Parameter | Value |",
        "|---|---|",
    ]
    for key, value in asdict(cfg).items():
        lines.append(f"| `{key}` | `{value}` |")

    lines += [
        "",
        "## Metrics",
        "",
        "| Metric | Value |",
        "|---|---|",
        f"| Precision | {val_metrics['precision']:.4f} |",
        f"| Recall | {val_metrics['recall']:.4f} |",
        f"| F1 | {val_metrics['f1']:.4f} |",
        f"| mAP50 | {val_metrics['mAP50']:.4f} |",
        f"| mAP50-95 | {val_metrics['mAP50-95']:.4f} |",
        "",
        f"**Target (mAP50-95 >= 0.95):** {'MET' if val_metrics['mAP50-95'] >= 0.95 else 'NOT MET'}",
        "",
        "## Training Duration",
        "",
        f"{int(hours)}h {int(minutes)}m {int(seconds)}s ({training_duration_sec:.1f}s total)",
        "",
        "## Model Size",
        "",
        f"`best.pt`: {model_size_mb:.2f} MB",
        "",
        "## Inference Speed",
        "",
        "| Device | Preprocess (ms) | Inference (ms) | Postprocess (ms) | Total mean (ms) | Total p95 (ms) | Meets 200ms target |",
        "|---|---|---|---|---|---|---|",
    ]
    for device, b in benchmarks.items():
        lines.append(
            f"| {device} | {b.preprocess_ms_mean:.1f} | {b.inference_ms_mean:.1f} | "
            f"{b.postprocess_ms_mean:.1f} | {b.total_ms_mean:.1f} | {b.total_ms_p95:.1f} | "
            f"{'Yes' if b.meets_200ms_target else 'No'} |"
        )

    return "\n".join(lines) + "\n"


def save_training_report(report_markdown: str, output_path: Path) -> Path:
    """Write the training report to disk.

    Parameters
    ----------
    report_markdown : str
        Rendered Markdown content.
    output_path : Path
        Destination file path.

    Returns
    -------
    Path
        The path written to.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(report_markdown)
    logger.info(f"Training report saved to {output_path}")
    return output_path


REPORT_MARKDOWN = build_training_report_markdown(
    CFG, ENV_INFO, VAL_METRICS, TRAINING_DURATION_SEC, BEST_WEIGHTS, BENCHMARK_RESULTS
)
TRAINING_REPORT_PATH = save_training_report(REPORT_MARKDOWN, REPORT_DIR / "training_report.md")
print(REPORT_MARKDOWN)


## 16. Save Experiment Metadata

In [ ]:
def get_git_metadata() -> Dict[str, Optional[str]]:
    """Collect git-friendly metadata about the current repository state.

    Returns
    -------
    Dict[str, Optional[str]]
        Commit hash, branch name, and dirty-state flag; ``None`` values
        indicate the command failed (e.g. not inside a git repository).
    """
    def _run(cmd: List[str]) -> Optional[str]:
        try:
            return subprocess.check_output(cmd, stderr=subprocess.DEVNULL).decode().strip()
        except Exception:  # noqa: BLE001
            return None

    return {
        "commit": _run(["git", "rev-parse", "HEAD"]),
        "branch": _run(["git", "rev-parse", "--abbrev-ref", "HEAD"]),
        "is_dirty": _run(["git", "status", "--porcelain"]) not in (None, ""),
    }


def build_experiment_metadata(
    cfg: TrainingConfig,
    val_metrics: Dict[str, float],
    training_duration_sec: float,
    export_results: List[ExportResult],
    benchmarks: Dict[str, LatencyBenchmark],
    weights_path: Path,
) -> Dict[str, Any]:
    """Assemble the machine-readable experiment metadata dictionary.

    Parameters
    ----------
    cfg : TrainingConfig
        Training configuration used for this run.
    val_metrics : Dict[str, float]
        Output of ``run_validation`` (the ``"_raw"`` key is dropped).
    training_duration_sec : float
        Wall-clock training duration, in seconds.
    export_results : List[ExportResult]
        Output of ``export_model``.
    benchmarks : Dict[str, LatencyBenchmark]
        Per-device latency benchmark results.
    weights_path : Path
        Path to the best checkpoint.

    Returns
    -------
    Dict[str, Any]
        Full experiment metadata, ready for JSON serialization.
    """
    metrics_clean = {k: v for k, v in val_metrics.items() if k != "_raw"}
    return {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "experiment_name": cfg.experiment_name,
        "hyperparameters": {k: v for k, v in asdict(cfg).items() if k not in ("data_yaml", "project_dir")},
        "metrics": metrics_clean,
        "training_duration_sec": training_duration_sec,
        "model_size_mb": round(weights_path.stat().st_size / (1024 * 1024), 3) if weights_path.is_file() else None,
        "exports": [vars(r) for r in export_results],
        "benchmarks": {k: vars(v) for k, v in benchmarks.items()},
        "git": get_git_metadata(),
    }


EXPERIMENT_METADATA = build_experiment_metadata(
    CFG, VAL_METRICS, TRAINING_DURATION_SEC, EXPORT_RESULTS, BENCHMARK_RESULTS, BEST_WEIGHTS
)

EXPERIMENT_JSON_PATH = REPORT_DIR / "experiment.json"
with open(EXPERIMENT_JSON_PATH, "w") as f:
    json.dump(EXPERIMENT_METADATA, f, indent=2, default=str)

logger.info(f"Experiment metadata saved to {EXPERIMENT_JSON_PATH}")
EXPERIMENT_METADATA


## Summary

- Best checkpoint: `RUN_DIR / "weights" / "best.pt"`
- Validation metrics, training curves, confusion matrix, and PR curves are reviewed above.
- Exported formats: TorchScript and ONNX (TensorRT if available) under the run directory.
- CPU/GPU latency benchmarks, split into preprocess/inference/postprocess, are recorded
  in `reports/training_report.md` and `reports/experiment.json`.
- Check both target metrics before moving on: **mAP50-95 ≥ 0.95** and **total inference
  latency ≤ 200ms**. If either target isn't met, revisit Section 7's hyperparameters
  (e.g. more epochs, adjusted augmentation, larger/smaller image size) and re-run from
  Section 9.
- Next notebook: `03_deploy_fastapi.ipynb`, wrapping the exported model in a FastAPI
  inference service.
